# DeepSeek: сравнение transfer pruning и Torch-Pruning

Первый, средний и последний слои сравниваются в одном запуске через `ComparePrunedModelsUsecase`.
Сравниваем активации, без градиентов. Usecase структурно изменяет модель: перед повторным запуском сравнения загрузите её заново.


In [1]:
import sys
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pruning_lens").is_dir())
sys.path.insert(0, str(ROOT))

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformer_lens.model_bridge import TransformerBridge
from lora_transfer_pruning.core.prune_task_type import GroupPruneTask, ModelPruneTask
from pruning_lens.usecase.compare_pruned_models_usecase import ComparePrunedModelsUsecase

MODEL = "deepseek-ai/DeepSeek-V2-Lite-Chat"
DEVICE = "cuda:3"
DTYPE = torch.float16

model = AutoModelForCausalLM.from_pretrained(
    MODEL, trust_remote_code=False, device_map={"": DEVICE},
    dtype=DTYPE, attn_implementation="eager",
).eval()
bridge = TransformerBridge.boot_transformers(MODEL, hf_model=model, dtype=DTYPE)


Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


Loading weights:   0%|          | 0/351 [00:00<?, ?it/s]

In [2]:
tokenizer = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=False)
tokens = tokenizer(
    "The attention mechanism allows a language model to use information from earlier tokens.",
    return_tensors="pt",
).input_ids.to(DEVICE)


In [5]:
n_layers = len(bridge.blocks)
layers = [0, n_layers // 2, n_layers - 1]
prune_task = ModelPruneTask({
    f"blocks.{layer}.attn.q_proj": GroupPruneTask(cols=None, rows=[1, 2, 130])
    for layer in layers
})

bridge.set_use_hook_mlp_in(True)

comparison = ComparePrunedModelsUsecase.compare_transfer_and_torch_prunings_on_layers(
    bridge, tokens, prune_task, layers=layers, compute_gradient=False,
)


/glazkov-dev/LoRa-Transfer-Pruning/.venv/lib/python3.10/site-packages/torch_pruning/dependency/graph.py:390: UserWarning: Unwrapped parameters detected: ['model.layers.11._original_component.input_layernorm._original_component.weight', 'model.layers.21._original_component.post_attention_layernorm._original_component.weight', 'model.layers.22._original_component.self_attn._original_component.kv_a_layernorm._original_component.weight', 'model.layers.5._original_component.self_attn._original_component.kv_a_layernorm._original_component.weight', 'model.layers.8._original_component.self_attn._original_component.kv_a_layernorm._original_component.weight', 'model.layers.11._original_component.post_attention_layernorm._original_component.weight', 'model.layers.13._original_component.mlp._original_component.experts.down_proj', 'model.layers.17._original_component.self_attn._original_component.kv_a_layernorm._original_component.weight', 'model.layers.12._original_component.mlp._original_componen

NotImplementedError: Can't automatically map indices from k head to v, their shapes not equal (126 != 128). Please provide kv_lora_idxs manually.